In [1]:
# STEP 1: Fix the working directory.
# This notebook lives in research/, but the config files (config/config.yaml, params.yaml,
# schema.yaml) and artifacts/ are relative to the project root. So we move up one folder.
# The "if" makes it safe to re-run: it only moves up when we are still inside research/.
import os
if os.path.basename(os.getcwd()) == "research":
    os.chdir("../")
%pwd

'/Users/strava/Developer/personal/ml-project/ML-flow'

In [2]:
# STEP 2: Load the MLflow / DagsHub credentials.
# MLflow reads these three environment variables automatically:
#   MLFLOW_TRACKING_URI      -> where to send the experiment data (our DagsHub repo)
#   MLFLOW_TRACKING_USERNAME -> DagsHub username
#   MLFLOW_TRACKING_PASSWORD -> DagsHub access token (a secret, so it lives in .env, not in code)
# load_dotenv() reads the git-ignored .env file in the project root and sets them.
from dotenv import load_dotenv
load_dotenv()
# Fail early with a clear message if one is missing, instead of a confusing 401/403 later.
for var in ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    assert os.environ.get(var), f"{var} is not set; add it to .env"

In [3]:
# STEP 3: The entity: a plain container for everything the evaluation needs.
# frozen=True makes it read-only, so the settings cannot be changed by accident later.
# It carries no logic. It just groups the values so we pass one object around, not many.
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path          # folder where evaluation output goes (artifacts/model_evaluation)
    test_data_path: Path    # test.csv made by the data transformation stage
    model_path: Path        # model.joblib made by the model trainer stage
    all_params: dict        # the ElasticNet hyperparameters (alpha, l1_ratio) to log to MLflow
    metric_file_name: Path  # where to save the scores locally (metrics.json)
    target_column: str      # the column we predict ("quality")
    mlflow_uri: str         # the MLflow registry URL (our DagsHub repo)

In [4]:
# STEP 4: The configuration manager: reads the YAML files and builds the entity from STEP 3.
# This keeps paths and settings out of the evaluation code, so changing them means editing
# config.yaml / params.yaml, not Python.
from wineQualityML.constants import *   # gives us CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from wineQualityML.utils.common import read_yaml, create_directories, save_json

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        # Load the three YAML files: paths (config), hyperparameters (params), columns (schema)
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        # Make sure the top-level artifacts/ folder exists
        create_directories([self.config.artifacts_root])


    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation   # the model_evaluation block of config.yaml
        params = self.params.ElasticNet         # alpha and l1_ratio from params.yaml
        schema =  self.schema.TARGET_COLUMN     # the target column definition from schema.yaml

        # Create artifacts/model_evaluation/ so we can save metrics.json into it
        create_directories([config.root_dir])

        # Fill the entity with the values read above
        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path = config.model_path,
            all_params=params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            # Read from the environment (.env) so it points at OUR DagsHub repo.
            # (It used to point at the tutorial author's repo, which gave a 403.)
            mlflow_uri=os.environ["MLFLOW_TRACKING_URI"],

        )

        return model_evaluation_config

In [5]:
# STEP 5: Imports for the evaluation itself.
import os
import pandas as pd                      # read test.csv
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score  # the three scores
from urllib.parse import urlparse        # to check if the MLflow URL is a local folder or a server
import mlflow
import mlflow.sklearn                    # MLflow helpers specific to scikit-learn models
import numpy as np
import joblib                            # load the saved model file

/Users/strava/Developer/personal/ml-project/ML-flow/venv/lib/python3.11/site-packages/mlflow/utils/requirements_utils.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [6]:
# STEP 6: The component: does the real work (score the model and log to MLflow).
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config


    def eval_metrics(self,actual, pred):
        # Three standard regression scores comparing true quality vs predicted quality:
        rmse = np.sqrt(mean_squared_error(actual, pred))  # typical error size, punishes big misses
        mae = mean_absolute_error(actual, pred)           # average absolute error, easy to read
        r2 = r2_score(actual, pred)                       # 1.0 = perfect, 0 = no better than guessing the mean
        return rmse, mae, r2



    def log_into_mlflow(self):

        # Load the unseen test data and the model trained in the previous stage
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        # Split into inputs (all columns except the target) and the true answers
        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]


        # Tell MLflow where the model registry lives (our DagsHub repo)
        mlflow.set_registry_uri(self.config.mlflow_uri)
        # "file" means MLflow is logging to a local folder; "https" means a remote server like DagsHub
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme


        # A "run" is one recorded experiment. Everything logged inside this block belongs to it.
        with mlflow.start_run():

            # Predict wine quality for the test rows, then score the predictions
            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

            # Saving metrics as local
            # Keep a copy of the scores on disk too (artifacts/model_evaluation/metrics.json)
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            # Log the hyperparameters used (alpha, l1_ratio) so each run records how its model was built
            mlflow.log_params(self.config.all_params)

            # Log the scores so runs can be compared side by side in the DagsHub UI
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)


            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                # Remote server: upload the model AND register it as "ElasticnetModel".
                # Each run adds a new numbered version of that registered model.
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                # Local folder: the registry is not available, so just store the model with the run
                mlflow.sklearn.log_model(model, "model")


In [7]:
# STEP 7: Run it: build the config, create the evaluator, score the model and log to MLflow.
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    # (the variable is reused: first the config entity, then the evaluator that wraps it)
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-10-04 21:13:09,249: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-10-04 21:13:09,251: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-04 21:13:09,252: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-10-04 21:13:09,253: INFO: common: created directory at: artifacts]
[2026-10-04 21:13:09,253: INFO: common: created directory at: artifacts/model_evaluation]
[2026-10-04 21:13:10,472: INFO: common: json file saved at: artifacts/model_evaluation/metrics.json]


/Users/strava/Developer/personal/ml-project/ML-flow/venv/lib/python3.11/site-packages/_distutils_hack/__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
Successfully registered model 'ElasticnetModel'.
2026/10/04 21:13:17 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: ElasticnetModel, version 1
Created version '1' of model 'ElasticnetModel'.
